In [ ]:
import sys
print(sys.version)

In [ ]:
pip install opencv-python mediapipe

In [ ]:
import mediapipe as mp
print(mp.__version__)

In [ ]:
# ====================================================================
# PROGRAMA: LUZ POR DEDO
#
# Descripcion:
# Cada dedo levantado enciende una luz virtual de color en pantalla.
# Usa MediaPipe Hands para detectar los 5 dedos individualmente.
#
# Mapeo dedo -> luz:
#   Pulgar  -> LUZ BLANCA
#   Indice  -> LUZ ROJA
#   Medio   -> LUZ VERDE
#   Anular  -> LUZ AZUL
#   Menique -> LUZ AMARILLA
#
# Controles: ESC para salir.
# ====================================================================
import cv2
import mediapipe as mp
import numpy as np

# --- INICIALIZACION DE MEDIAPIPE ---
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1,
                       min_detection_confidence=0.7, min_tracking_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

# --- INICIALIZACION DE LA CAMARA ---
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# --- CONFIGURACION DE LUCES ---
# Cada dedo tiene: nombre, color (BGR), posicion en pantalla
LUCES = [
    {"nombre": "Pulgar",  "color": (255, 255, 255), "pos": (100, 80)},   # Blanca
    {"nombre": "Indice",  "color": (0, 0, 255),    "pos": (210, 80)},   # Roja
    {"nombre": "Medio",   "color": (0, 255, 0),    "pos": (320, 80)},   # Verde
    {"nombre": "Anular",  "color": (255, 0, 0),    "pos": (430, 80)},   # Azul
    {"nombre": "Menique", "color": (0, 255, 255),  "pos": (540, 80)},   # Amarilla
]

def detectar_dedos_individuales(hand_landmarks):
    """
    Detecta que dedos especificos estan levantados.
    Retorna una lista de 5 booleanos en orden:
    [pulgar, indice, medio, anular, menique]
    """
    lm = hand_landmarks.landmark
    dedos = []

    # Pulgar (ID 4 vs ID 3): coordenada x
    dedos.append(1 if lm[4].x < lm[3].x else 0)

    # Indice  (ID 8 vs ID 6)
    dedos.append(1 if lm[8].y < lm[6].y else 0)

    # Medio   (ID 12 vs ID 10)
    dedos.append(1 if lm[12].y < lm[10].y else 0)

    # Anular  (ID 16 vs ID 14)
    dedos.append(1 if lm[16].y < lm[14].y else 0)

    # Menique (ID 20 vs ID 18)
    dedos.append(1 if lm[20].y < lm[18].y else 0)

    return dedos

def dibujar_luz(frame, centro, color_bgr, encendida, nombre):
    """Dibuja un circulo como luz (encendida o apagada) con etiqueta."""
    radio = 30
    if encendida:
        # Brillo completo + resplandor
        cv2.circle(frame, centro, radio + 10, color_bgr, cv2.FILLED)
        cv2.circle(frame, centro, radio, (255, 255, 255), cv2.FILLED)
        cv2.circle(frame, centro, radio, color_bgr, 3)
        cv2.putText(frame, "ENCENDIDA", (centro[0] - 40, centro[1] + radio + 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color_bgr, 2)
    else:
        # Apagada: circulo gris oscuro
        cv2.circle(frame, centro, radio, (50, 50, 50), cv2.FILLED)
        cv2.circle(frame, centro, radio, (100, 100, 100), 2)
        cv2.putText(frame, "APAGADA", (centro[0] - 35, centro[1] + radio + 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (100, 100, 100), 1)

    # Nombre del dedo arriba de la luz
    cv2.putText(frame, nombre, (centro[0] - 25, centro[1] - radio - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 2)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    height, width, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_frame)

    # Estado inicial: todas las luces apagadas
    dedos_estado = [0, 0, 0, 0, 0]

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        dedos_estado = detectar_dedos_individuales(hand_landmarks)

        # Mostrar texto con que dedos estan levantados
        nombres_levantados = []
        for i, estado in enumerate(dedos_estado):
            if estado:
                nombres_levantados.append(LUCES[i]["nombre"])
        texto = "Dedos levantados: " + (", ".join(nombres_levantados) if nombres_levantados else "ninguno")
        cv2.putText(frame, texto, (10, height - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    else:
        cv2.putText(frame, "Sin mano detectada", (10, height - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (100, 100, 100), 2)

    # Dibujar las 5 luces con su estado
    for i, luz in enumerate(LUCES):
        dibujar_luz(frame, luz["pos"], luz["color"], dedos_estado[i], luz["nombre"])

    # Instrucciones
    cv2.putText(frame, "ESC para salir", (width - 150, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)

    cv2.imshow("Luz por Dedo", frame)

    if cv2.waitKey(1) & 0xFF == 27 or cv2.getWindowProperty("Luz por Dedo", cv2.WND_PROP_VISIBLE) < 1:
        break

cap.release()
cv2.destroyAllWindows()